# Visual Fixed-Lag Smoother Example with Smart Factors

A camera moves forward for 26 frames and tracks landmarks for up to 20 frames (the visual odometry dataset of `StereoVOExample_large.ipynb`). Every track is one `SmartProjectionPoseFactor` on the left image, the visual odometry poses of the dataset provide the odometry, and an `IncrementalFixedLagSmoother` with a lag of 5 frames estimates the trajectory.

A factor inside the smoother is never modified: a copy with the new measurement replaces it through `factorsToRemove`. A track that outlives the window is marginalized with the leaving pose (`MARGINALIZE`, the default) or continued with that pose held fixed (`CONDITION`). Both are compared with the full batch solution; the mathematics is in [SmartFactorsFixedLag.md](https://github.com/borglab/gtsam/blob/develop/gtsam/slam/doc/SmartFactorsFixedLag.md), and the C++ version is `examples/VisualFixedLagSmootherExample_SmartFactor.cpp`.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/nonlinear/VisualFixedLagSmootherExample_SmartFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import numpy as np
import plotly.graph_objects as go

import gtsam
from gtsam.symbol_shorthand import X

## 1. Dataset

The left camera calibration, one visual odometry pose per frame, and the left pixel $(u_L, v)$ of every observation, grouped by frame.

In [3]:
fx, fy, s, u0, v0, baseline = np.loadtxt(gtsam.findExampleDataFile("VO_calibration.txt"))
K = gtsam.Cal3_S2(fx, fy, s, u0, v0)

vo_poses = {}
for row in np.loadtxt(gtsam.findExampleDataFile("VO_camera_poses_large.txt")):
    vo_poses[int(row[0])] = gtsam.Pose3(row[1:].reshape(4, 4))

frames = {}  # frame -> list of (landmark, pixel)
for row in np.loadtxt(gtsam.findExampleDataFile("VO_stereo_factors_large.txt")):
    frames.setdefault(int(row[0]), []).append((int(row[1]), row[[2, 4]]))
first, last = min(frames), max(frames)

## 2. Frame factors and the batch reference

Each frame adds the prior or the odometry factor of its pose, initialized from the previous estimate. The reference for frame $i$ is the full batch solution over frames $1 \ldots i$ with one smart factor per landmark.

In [4]:
pixel_noise = gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
odometry_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01, 0.01, 0.01, 0.1, 0.1, 0.1]))
prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.001, 0.001, 0.001, 0.01, 0.01, 0.01]))


def add_frame(i, previous, graph, values):
    if i == first:
        graph.push_back(gtsam.PriorFactorPose3(X(i), vo_poses[i], prior_noise))
        values.insert(X(i), vo_poses[i])
    else:
        odometry = vo_poses[i - 1].between(vo_poses[i])
        graph.push_back(gtsam.BetweenFactorPose3(X(i - 1), X(i), odometry, odometry_noise))
        values.insert(X(i), previous.compose(odometry))


def smart_factors(up_to):
    """One smart factor per landmark seen at least twice in frames first ... up_to."""
    factors = {}
    for i in range(first, up_to + 1):
        for landmark, pixel in frames[i]:
            if landmark not in factors:
                factors[landmark] = gtsam.SmartProjectionPoseFactorCal3_S2(pixel_noise, K)
            factors[landmark].add(pixel, X(i))
    return {landmark: factor for landmark, factor in factors.items() if factor.size() >= 2}


def full_batch(up_to):
    graph, initial = gtsam.NonlinearFactorGraph(), gtsam.Values()
    for i in range(first, up_to + 1):
        add_frame(i, initial.atPose3(X(i - 1)) if i > first else None, graph, initial)
    for factor in smart_factors(up_to).values():
        graph.push_back(factor)
    return gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()


batch_positions = {i: full_batch(i).atPose3(X(i)).translation() for i in range(first, last + 1)}

## 3. Fixed-lag smoothing with the replace protocol

A `Track` holds the factor of one landmark and its slot in the smoother, or -1 until it has two measurements. Every frame, a copy of the factor with the new measurement replaces the one in the smoother; the slots of the copies come back in `getNewFactorsIndices()`, followed by the slots of the conditioned copies the smoother made of the factors in `getConditionedFactorIndices()`. A slot that is empty afterwards means the factor was consumed by marginalization, and its track starts afresh. After every update, `run` also records the latest estimate of every pose and the point every smart factor in the smoother triangulates, for the plot in section 5.

In [5]:
class Track:
    def __init__(self):
        self.factor, self.slot = None, -1


def run(mode):
    params = gtsam.ISAM2Params()
    params.findUnusedFactorSlots = True  # reuse the slots of removed factors
    smoother = gtsam.IncrementalFixedLagSmoother(5.0, params)  # lag of 5 frames
    smoother.setMarginalizationMode(mode)
    tracks, rows, trajectory, landmarks = {}, [], gtsam.Values(), {}
    print("frame  poses  smart factors  longest track  |to batch| (m)")
    for i in range(first, last + 1):
        new_factors, new_values = gtsam.NonlinearFactorGraph(), gtsam.Values()
        add_frame(i, smoother.calculateEstimatePose3(X(i - 1)) if i > first else None,
                  new_factors, new_values)

        to_remove, pending = [], []  # pending: (landmark, index in new_factors)
        graph = smoother.getFactors()
        for landmark, pixel in frames[i]:
            track = tracks.setdefault(landmark, Track())
            in_smoother = (track.slot >= 0 and graph.exists(track.slot)
                           and graph.at(track.slot).equals(track.factor, 1e-12))
            if track.slot >= 0 and not in_smoother:
                track.factor = None
            if track.factor is None:
                factor = gtsam.SmartProjectionPoseFactorCal3_S2(pixel_noise, K)
            else:
                factor = track.factor.clone()
            factor.add(pixel, X(i))
            if in_smoother:
                to_remove.append(track.slot)
            track.factor, track.slot = factor, -1
            if factor.size() >= 2:
                pending.append((landmark, new_factors.size()))
                new_factors.push_back(factor)
        ours = new_factors.size()
        result = smoother.update(new_factors, new_values, {X(i): float(i)}, to_remove)

        new_slots = result.getNewFactorsIndices()
        for landmark, index in pending:
            tracks[landmark].slot = new_slots[index]
        by_slot = {track.slot: track for track in tracks.values() if track.slot >= 0}
        for k, old_slot in enumerate(result.getConditionedFactorIndices()):
            if old_slot in by_slot:
                by_slot[old_slot].slot = new_slots[ours + k]
        graph = smoother.getFactors()
        for track in tracks.values():
            if track.slot < 0:
                continue
            factor = graph.at(track.slot) if graph.exists(track.slot) else None
            if isinstance(factor, gtsam.SmartProjectionPoseFactorCal3_S2):
                track.factor = factor
            else:
                track.factor, track.slot = None, -1

        smart = [graph.at(slot) for slot in range(graph.size()) if graph.exists(slot)]
        smart = [f for f in smart if isinstance(f, gtsam.SmartProjectionPoseFactorCal3_S2)]
        longest = max((f.dim() // 2 for f in smart), default=0)
        # The latest estimate of every pose, and the point every smart factor in the smoother
        # triangulates; a copy triangulates, so the smoother's factor keeps its cache.
        estimate = smoother.calculateEstimate()
        trajectory.insert_or_assign(estimate)
        for landmark, track in tracks.items():
            if track.slot >= 0:
                point = track.factor.clone().point(estimate)
                if point.valid():
                    landmarks[landmark] = point.get()
        position = smoother.calculateEstimatePose3(X(i)).translation()
        to_batch = np.linalg.norm(position - batch_positions[i])
        rows.append((i, len(smoother.timestamps()), len(smart), longest, to_batch))
        print(f"{i:5}{rows[-1][1]:7}{len(smart):15}{longest:15}{to_batch:16.3f}")
    return rows, trajectory, landmarks

## 4. Both modes

With `MARGINALIZE` no track is longer than the six poses of the window, and the estimate stays within millimetres of the batch solution: exact marginalization keeps the correlation between the leaving pose and the window. With `CONDITION` the tracks keep growing, up to the window plus `SmartProjectionParams.maxFixedCameras` fixed cameras, and the estimate drifts a few centimetres away, because a fixed camera keeps the estimation error the pose had when it left the window. On a scene where a few landmarks stay in view the whole time the comparison reverses, since consuming every factor at once cuts all tracks; `MARGINALIZE` is the right default.

In [6]:
modes = gtsam.FixedLagSmoother.MarginalizationMode
results, trajectories, landmark_estimates = {}, {}, {}
for name, mode in [("MARGINALIZE", modes.MARGINALIZE), ("CONDITION", modes.CONDITION)]:
    print(name)
    results[name], trajectories[name], landmark_estimates[name] = run(mode)
    print()

MARGINALIZE
frame  poses  smart factors  longest track  |to batch| (m)
    1      1              0              0           0.000
    2      2            224              2           0.000
    3      3            308              3           0.000
    4      4            380              4           0.000
    5      5            454              5           0.000
    6      6            543              6           0.000
    7      6            420              6           0.000
    8      6            435              6           0.001
    9      6            464              6           0.001
   10      6            496              6           0.001
   11      6            514              6           0.002
   12      6            510              6           0.003
   13      6            528              6           0.003
   14      6            532              6           0.003
   15      6            532              6           0.004
   16      6            546              6  

In [7]:
fig = go.Figure()
for name, rows in results.items():
    fig.add_trace(go.Scatter(x=[r[0] for r in rows], y=[r[4] for r in rows], name=name, mode="lines+markers"))
fig.update_layout(title="Distance of the newest pose from the full batch solution",
                  xaxis_title="frame", yaxis_title="m", height=400)
fig.show()

## 5. Trajectory and landmarks

The visual odometry poses of the dataset, the full batch solution, and the fixed-lag trajectories, in which every pose keeps its last estimate before it left the window. Smart factors have no landmark variables: a full batch landmark is the point its smart factor triangulates at the solution, and a fixed-lag landmark is the last point its smart factor in the smoother triangulated. Every landmark with an estimate is plotted, and the printed counts name the others: a landmark has no estimate when its triangulation fails, for example because the point would lie behind a camera. The fixed-lag estimate of a landmark within 20 m of the path agrees with the full batch to a few centimetres; the depth of a distant landmark is constrained much less, so its estimates differ by decimetres to metres, less so with `CONDITION`, whose tracks are longer, and their points overlap only partly. Hover over a point to see its landmark, click a legend entry to show or hide a trace, and scroll to zoom: some landmarks are well over 100 m away.

In [8]:
seen = {landmark for observations in frames.values() for landmark, _ in observations}
batch = full_batch(last)
triangulations = {landmark: factor.point(batch) for landmark, factor in smart_factors(last).items()}
failed = {landmark: point.status.name for landmark, point in triangulations.items() if not point.valid()}
print(f"{len(seen)} landmarks, {len(triangulations)} of them seen at least twice; "
      f"the full batch cannot triangulate {failed}")
estimates = {"Full batch": ("black", batch, {landmark: point.get() for landmark, point in triangulations.items()
                                             if point.valid()})}
for (name, trajectory), color in zip(trajectories.items(), ["royalblue", "crimson"]):
    estimates[f"Fixed-lag, {name}"] = (color, trajectory, landmark_estimates[name])

vo = np.array([vo_poses[i].translation() for i in range(first, last + 1)])
fig = go.Figure(go.Scatter3d(x=vo[:, 0], y=vo[:, 1], z=vo[:, 2], mode="lines", name="Visual odometry",
                             line=dict(color="gray", width=2, dash="dash")))
for name, (color, values, points) in estimates.items():
    ids = sorted(points)
    print(f"{name}: {len(ids)} of {len(seen)} landmarks plotted, no estimate for {sorted(seen - set(ids))}")
    poses = np.array([values.atPose3(X(i)).translation() for i in range(first, last + 1)])
    fig.add_trace(go.Scatter3d(x=poses[:, 0], y=poses[:, 1], z=poses[:, 2], mode="lines+markers",
                               name=name, line=dict(color=color, width=3), marker=dict(size=2, color=color)))
    xyz = np.array([points[landmark] for landmark in ids])
    fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                               name=f"{name}: landmarks", text=[f"landmark {landmark}" for landmark in ids],
                               marker=dict(size=1.5, color=color, opacity=0.5),
                               visible=True if name == "Full batch" else "legendonly"))
fig.update_layout(title="Trajectory and landmarks", height=600, legend=dict(x=0, y=1),
                  margin=dict(l=0, r=0, b=0, t=30),
                  scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)", aspectmode="data",
                             camera=dict(up=dict(x=0, y=-1, z=0), eye=dict(x=-1, y=-1, z=-1))))
fig.show()

2634 landmarks, 2634 of them seen at least twice; the full batch cannot triangulate {340: 'BEHIND_CAMERA', 346: 'BEHIND_CAMERA', 8209: 'BEHIND_CAMERA'}
Full batch: 2631 of 2634 landmarks plotted, no estimate for [340, 346, 8209]
Fixed-lag, MARGINALIZE: 2631 of 2634 landmarks plotted, no estimate for [340, 346, 8209]
Fixed-lag, CONDITION: 2631 of 2634 landmarks plotted, no estimate for [340, 346, 8209]
